# Lab 16-06 · Designing FIR Filters Four Ways

Six parts, each built on `firdesign.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import firdesign as fd

fs = 48000.0
wp, ws = 2 * np.pi * 4000 / fs, 2 * np.pi * 5000 / fs
dp, ds = fd.deltas(0.1, 60)
print(f"running example: dp = {dp:.6f}, ds = {ds:.4f}, transition = {ws - wp:.4f} rad/sample")

running example: dp = 0.005756, ds = 0.0010, transition = 0.1309 rad/sample


## Part 1 · Types and zeros

*Predict:* which of the four filters below has a zero at $z = -1$, and which at $z = +1$?

In [2]:
examples = {"I": [1, 2, 3, 2, 1], "II": [1, 3, 3, 1], "III": [1, 0, -1], "IV": [1, -3, 3, -1]}
w = np.linspace(0, np.pi, 1001)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for name, h in examples.items():
    assert fd.fir_type(h) == {"I": 1, "II": 2, "III": 3, "IV": 4}[name]
    A = fd.amplitude(h, w)
    ax[0].plot(w / np.pi, A, label=f"type {name}")
    z = np.roots(h)
    ax[1].plot(z.real, z.imag, "o", label=f"type {name}")
    # zeros come in reciprocal pairs (repeated roots are found only to ~1e-5, hence the tolerance)
    zr = 1 / np.conj(z)
    assert all(np.min(np.abs(z - r)) < 1e-4 for r in zr)
    print(f"type {name}: A(0) = {A[0]:+.3f}, A(pi) = {A[-1]:+.3f}, zeros {np.round(z, 3)}")
assert abs(fd.amplitude(examples["II"], np.pi)[0]) < 1e-12
assert abs(fd.amplitude(examples["III"], 0)[0]) < 1e-12 and abs(fd.amplitude(examples["III"], np.pi)[0]) < 1e-12
assert abs(fd.amplitude(examples["IV"], 0)[0]) < 1e-12
t = np.linspace(0, 2 * np.pi, 200)
ax[1].plot(np.cos(t), np.sin(t), "k", lw=0.5)
ax[1].set_aspect("equal")
ax[0].set_xlabel("ω/π"); ax[0].set_ylabel("A(ω)"); ax[0].legend(); ax[1].legend(fontsize=7)
plt.show()

type I: A(0) = +9.000, A(pi) = +1.000, zeros [-0.5+0.866j -0.5-0.866j -0.5+0.866j -0.5-0.866j]
type II: A(0) = +8.000, A(pi) = +0.000, zeros [-1.+0.j -1.-0.j -1.+0.j]
type III: A(0) = +0.000, A(pi) = +0.000, zeros [-1.  1.]
type IV: A(0) = +0.000, A(pi) = -8.000, zeros [1.+0.j 1.+0.j 1.-0.j]


## Part 2 · Gibbs

*Predict:* the overshoot of the truncated ideal half-band filter at $M$ = 20 and at $M$ = 400.

In [3]:
rows = []
for M in (20, 50, 100, 400):
    h = fd.ideal_lowpass(M, np.pi / 2)
    w = np.linspace(0.3 * np.pi, 0.5 * np.pi, 200001)
    A = fd.amplitude(h, w)
    i = int(np.argmax(A))
    rows.append((M, 100 * (A[i] - 1), (np.pi / 2 - w[i]), 2 * np.pi / (M + 1)))
for M, ov, off, pred in rows:
    print(f"M = {M:4d}: overshoot {ov:.2f} %, peak {off:.4f} rad below the cutoff (prediction {pred:.4f})")
assert abs(rows[0][1] - 9.12) < 0.005 and abs(rows[1][1] - 8.97) < 0.005 and abs(rows[3][1] - 8.95) < 0.005
assert all(8.9 < r[1] < 9.2 for r in rows)          # the overshoot does not shrink
assert rows[3][2] < rows[0][2] / 10                 # but it moves toward the cutoff

M =   20: overshoot 9.12 %, peak 0.3142 rad below the cutoff (prediction 0.2992)
M =   50: overshoot 8.97 %, peak 0.1208 rad below the cutoff (prediction 0.1232)
M =  100: overshoot 8.96 %, peak 0.0628 rad below the cutoff (prediction 0.0622)
M =  400: overshoot 8.95 %, peak 0.0157 rad below the cutoff (prediction 0.0157)


## Part 3 · The window table

*Predict:* will the Blackman design's error change when $M$ doubles?

In [4]:
def peak_error_db(name, M, wc):
    h = fd.ideal_lowpass(M, wc) * fd.window(name, M)
    w = np.linspace(0, np.pi, 100001)
    A = fd.amplitude(h, w)
    ic = np.searchsorted(w, wc)
    z = ic + np.argmax(A[ic:] <= 0)
    o = ic - np.argmax((A[:ic] >= 1)[::-1])
    return 20 * math.log10(max(np.max(np.abs(A[z:])), np.max(np.abs(A[:o] - 1))))

table = {"rect": -21.0, "hann": -43.9, "hamming": -54.2, "blackman": -75.3}
for name, ref in table.items():
    vals = [peak_error_db(name, M, np.pi / 2) for M in (50, 100, 200)]
    print(f"{name:9s}", " ".join(f"{v:7.2f}" for v in vals), f"  (chapter: {ref})")
    tol = 1.5 if name == "hamming" else 0.6   # Hamming's cancelling sidelobe makes it wander with M and the cutoff
    assert all(abs(v - ref) < tol for v in vals)
ham = [peak_error_db("hamming", 100, f * np.pi) for f in np.linspace(0.15, 0.85, 8)]
print("Hamming over cutoffs 0.15π to 0.85π:", np.round(ham, 2))
assert 1.0 < max(ham) - min(ham) < 3.0

rect       -20.94  -20.96  -20.96   (chapter: -21.0)


hann       -43.94  -43.94  -43.94   (chapter: -43.9)


hamming    -53.12  -54.22  -54.18   (chapter: -54.2)


blackman   -75.35  -75.28  -75.29   (chapter: -75.3)


Hamming over cutoffs 0.15π to 0.85π: [-51.83 -52.83 -53.36 -53.74 -53.74 -53.36 -52.83 -51.83]


## Part 4 · Kaiser designs, and repairing one

*Predict:* adding taps one at a time at a fixed $\\beta$, how many until the 70 dB design passes?

In [5]:
A = 60
beta = fd.kaiser_beta(A)
Mest = fd.kaiser_order(A, ws - wp)
h = fd.window_lowpass(174, wp, ws, "kaiser", beta)
m = fd.measure(h, wp, ws)
print(f"Example 16-06.3: beta = {beta:.3f}, M = {Mest:.2f} -> 174; ripple {m[2]:.4f} dB pp, attenuation {m[3]:.2f} dB")
assert abs(m[3] - 60.21) < 0.005 and m[0] <= dp

wp2, ws2 = 0.3 * np.pi, 0.35 * np.pi
d70 = 10 ** (-3.5)
b70 = fd.kaiser_beta(70)
print(f"70 dB: beta = {b70:.3f}, M estimate = {fd.kaiser_order(70, ws2 - wp2):.2f}")
for M in range(173, 200):
    mm = fd.measure(fd.window_lowpass(M, wp2, ws2, "kaiser", b70), wp2, ws2)
    if mm[0] <= d70 and mm[1] <= d70:
        break
print(f"same beta: first passing order {M} ({mm[3]:.2f} dB)")
assert M == 192
b71 = fd.kaiser_beta(71)
M71 = math.ceil(fd.kaiser_order(71, ws2 - wp2))
mm = fd.measure(fd.window_lowpass(M71, wp2, ws2, "kaiser", b71), wp2, ws2)
print(f"redesign for 71 dB: beta = {b71:.3f}, M = {M71}, attenuation {mm[3]:.2f} dB")
assert M71 == 176 and mm[0] <= d70 and mm[1] <= d70

Example 16-06.3: beta = 5.653, M = 173.85 -> 174; ripple 0.0169 dB pp, attenuation 60.21 dB
70 dB: beta = 6.755, M estimate = 172.74


same beta: first passing order 192 (70.10 dB)


redesign for 71 dB: beta = 6.865, M = 176, attenuation 70.24 dB


## Part 5 · Frequency sampling

*Predict:* is the best transition sample nearer 0.1, 0.4 or 0.7?

In [6]:
N, kp = 33, 4
T = np.linspace(0, 1, 201)
def stop_peak(t):
    h = fd.freq_sampling_lowpass(N, kp, t)
    w = np.linspace(2 * np.pi * (kp + 2) / N, np.pi, 20000)
    return 20 * math.log10(np.max(np.abs(fd.amplitude(h, w))))
peaks = np.array([stop_peak(t) for t in T])
ib = int(np.argmin(peaks))
print(f"T = 0: {peaks[0]:.1f} dB; best T on the grid = {T[ib]:.3f}: {peaks[ib]:.1f} dB")
assert abs(peaks[0] + 20.5) < 0.05 and abs(T[ib] - 0.393) < 0.006 and peaks[ib] < -42.0
h = fd.freq_sampling_lowpass(N, kp, 0.393)
k = np.arange(17)
assert np.allclose(fd.amplitude(h, 2 * np.pi * k / N), [1, 1, 1, 1, 1, 0.393] + [0] * 11, atol=1e-12)
plt.figure(figsize=(6, 3)); plt.plot(T, peaks); plt.xlabel("transition sample T"); plt.ylabel("stopband peak (dB)"); plt.show()

T = 0: -20.5 dB; best T on the grid = 0.395: -42.1 dB


## Part 6 · Equiripple, and a real signal

*Predict:* the delay, in samples, of the shortest equiripple filter for the running example.

In [7]:
n_min, h = fd.min_taps(lambda n: fd.remez_lowpass(n, wp, ws, dp, ds), wp, ws, dp, ds, 120, 160)
m = fd.measure(h, wp, ws)
alt = fd.alternations(h, wp, ws, dp, ds)
print(f"shortest: {n_min} taps, ripple {m[2]:.4f} dB pp, attenuation {m[3]:.2f} dB, {alt} alternations")
assert n_min == 137 and alt == (n_min - 1) // 2 + 2

n = np.arange(48000)
x = np.cos(2 * np.pi * 3000 * n / fs) + np.cos(2 * np.pi * 6000 * n / fs)
y = signal.lfilter(h, 1, x)
seg = slice(24000, 48000)
def tone_amp(sig, f):
    t = n[seg]
    a = 2 * np.mean(sig[seg] * np.cos(2 * np.pi * f * t / fs))
    b = 2 * np.mean(sig[seg] * np.sin(2 * np.pi * f * t / fs))
    return math.hypot(a, b)
g3 = 20 * math.log10(tone_amp(y, 3000))
a6 = -20 * math.log10(tone_amp(y, 6000))
print(f"3 kHz gain {g3:.3f} dB, 6 kHz attenuation {a6:.2f} dB")
assert abs(g3 + 0.048) < 0.0005 and abs(a6 - 74.09) < 0.005 and a6 >= 60

nb = np.arange(6000)
burst = np.exp(-0.5 * ((nb - 2000) / 200) ** 2) * np.cos(2 * np.pi * 2000 * nb / fs)
out = signal.lfilter(h, 1, burst)
ex, ey = np.abs(signal.hilbert(burst)), np.abs(signal.hilbert(out))
delay = int(np.argmax(ey) - np.argmax(ex))
print(f"burst delay {delay} samples = {1000 * delay / fs:.3f} ms")
assert delay == (n_min - 1) // 2 == 68
plt.figure(figsize=(7, 3)); plt.plot(nb, burst, lw=0.6, label="input"); plt.plot(nb, out, lw=0.6, label="output")
plt.xlim(1200, 2900); plt.legend(); plt.xlabel("sample"); plt.show()

shortest: 137 taps, ripple 0.0958 dB pp, attenuation 60.36 dB, 70 alternations
3 kHz gain -0.048 dB, 6 kHz attenuation 74.09 dB
burst delay 68 samples = 1.417 ms
